In [ ]:
!pip install gensim  # NOTA: o Gensim ainda usa uma versão desatualizada do Numpy (<2)
                     #        por isso, para funcionar, vai ser necessário fazer
                     #        o downgrade do numpy e REINICIAR o ambiente antes
                     #        de executar esse script

#Exercícios Complementares
Carregue o dataset a seguir, também utilizado em exercícios anteriores. Vale destacar que nesse dataset:
- o texto dos documentos que compõem o córpus está inserido no atributo title-abstract;
- o rótulo dos documentos está definido no atributo label.

In [ ]:
import requests, os


url = f'https://raw.githubusercontent.com/watinha/nlp-text-mining-datasets/main/slr.csv'

def download (url):
  filename = url.split('/')[-1]

  if (os.path.isfile(f'./{filename}')):
    print('Arquivo já existente no Runtime... Tudo OK')
    return

  response = requests.get(url)
  with open(f'./{filename}', 'wb') as f:
      f.write(response.content)
      print('Download realizado e arquivo extraído no Runtime... Tudo OK')

download(url)



Download realizado e arquivo extraído no Runtime... Tudo OK


##Treinamento e Teste de um Modelo LSTM
Modele, treine (10 épocas) e teste um classificador com uma camada de Embeddings de 300 dimensões, 2 camadas de LSTM com 300 neurônios, uma camada Densa de 300 neurônios e uma camada Densa com um único neurônio (saída). No teste, avalie a acurácia geral do modelo e as métricas de Precision, Recall e F-Score para as classes. Utilize um máximo de 1000 termos para o vocabulário e sequências de até 200 tokens de tamanho.


In [ ]:
import pandas as pd, numpy as np

from sklearn.model_selection import train_test_split


df = pd.read_csv('slr.csv')
corpus = df['title-abstract']
labels = df['label']

print(df['label'].value_counts())
X_train, X_test, y_train, y_test = train_test_split(corpus, labels, random_state=42)
print(X_train.head())

X_train = np.array(X_train)
X_test = np.array(X_test)
y_train = np.array(y_train)
y_test = np.array(y_test)

label
1    23
0    14
Name: count, dtype: int64
8     Overlooked aspects of COTS-based development\n...
15    The consistency of empirical comparisons of re...
12    The impact of software engineering research on...
32    A systematic review of theory use in software ...
9     An analysis of research in computing disciplin...
Name: title-abstract, dtype: object


In [ ]:
from keras.layers import TextVectorization, Embedding, LSTM, Dense, Input
from keras.models import Sequential


EMBEDDING_DIM = 300
NEURONS = 300
SEQUENCE_LENGTH = 200

vectorization_layer = TextVectorization(output_sequence_length=SEQUENCE_LENGTH)
vectorization_layer.adapt(X_train)

model = Sequential()
model.add(vectorization_layer)
model.add(Embedding(len(vectorization_layer.get_vocabulary()), output_dim=NEURONS))
model.add(LSTM(NEURONS, return_sequences=True))
model.add(LSTM(NEURONS))
model.add(Dense(NEURONS, activation='relu'))
model.add(Dense(1, activation='sigmoid'))
model.build(input_shape=(None, 1))
model.summary()

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

model.fit(X_train, y_train, epochs=10)

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization_1            │ (None, 200)            │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_1 (Embedding)         │ (None, 200, 300)       │       326,700 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 200, 300)       │       721,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 300)            │       721,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 300)            │        90,300 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           301 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,859,701 (7.09 MB)

 Trainable params: 1,859,701 (7.09 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 6s 6s/step - accuracy: 0.4074 - loss: 0.6932
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.5926 - loss: 0.6835
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.5926 - loss: 0.6876
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.6296 - loss: 0.6789
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.6296 - loss: 0.6796
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.6296 - loss: 0.6760
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.6296 - loss: 0.6710
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.6296 - loss: 0.6674
Epoch 9/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.6296 - loss: 0.6570
Epoch 10/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.6296 - loss: 0.6421


In [ ]:
from sklearn.metrics import classification_report


y_pred = model.predict(X_test)
y_pred = (y_pred > 0.5).astype('int')

print(classification_report(y_test, y_pred))

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 876ms/step
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         2
           1       0.71      0.62      0.67         8

    accuracy                           0.50        10
   macro avg       0.36      0.31      0.33        10
weighted avg       0.57      0.50      0.53        10



##Embeddings Pré-treinados
O dataset contém documentos escritos em inglês. Para tentar melhorar a acurácia do nosso modelo, vamos carregar Word Embeddings Pré-treinados e utiliza-lo para inicializar os pesos da camada de Embeddings do nosso modelo. A sugestão é utilizar o modelo glove carregado pelo código a seguir.

In [ ]:
import zipfile

from gensim.scripts.glove2word2vec import glove2word2vec


def unzip (filename):
  with open(filename, 'rb') as f:
    with zipfile.ZipFile(f, 'r') as zip_ref:
      zip_ref.extractall('./')
      print("Arquivo pronto...")

download('https://nlp.stanford.edu/data/glove.6B.zip')
unzip('glove.6B.zip')

glove2word2vec('./glove.6B.300d.txt', './glove.6B.300d.w2v.txt')


Implemente uma função para carregar os valores dos vetores a partir do arquivo glove.6B.300d.w2v.txt. Utilize esses vetores, juntamente com o vocabulário dos embeddings pré-treinados para treinar um modelo de classificação.

Nesse modelo de classificação, utilizem a seguinte configuração:
1. Uma camada de TextVectorization configurada com o vocabulário dos Embeddings carregados;
2. Uma camada de Embeddings com os pesos inicializados com os Embeddings do arquivo glove.6B.300d.w2v.txt;
3. Uma camada Conv1D 64 filtros e tamanho de kernel de 3;
4. Uma camada de MaxPooling1D com pool_size de 3;
5. Duas camadas de LSTM com 300 neurônios;
6. Uma camada Densa de 300 neurônios;
7. Uma camada Densa de 1 neurônio de saída.

In [ ]:
import numpy as np

from gensim.models import KeyedVectors


vectors = KeyedVectors.load_word2vec_format('./glove.6B.300d.w2v.txt')

def generate_weight_matrix (vectors, dim=50):
  vectors_vocabulary = vectors.key_to_index.keys()
  vocabulary = ['', '[UNK]'] + list(vectors_vocabulary)
  weights_matrix = np.zeros((len(vocabulary), dim))
  for i, word in enumerate(vocabulary):
    if word in vectors_vocabulary:
      weights_matrix[i] = vectors[word]
    else:
      weights_matrix[i] = np.random.rand(dim)
  return (vocabulary, weights_matrix)


(vocabulary, weights_matrix) = generate_weight_matrix(
    vectors, dim=EMBEDDING_DIM)

print(weights_matrix)



In [ ]:
from keras.layers import Conv1D, MaxPooling1D


FILTERS = 64
KERNEL_SIZE = 3
POOL_SIZE = 3

model = Sequential()
model.add(TextVectorization(output_sequence_length=SEQUENCE_LENGTH, vocabulary=vocabulary))
model.add(Embedding(len(vocabulary), output_dim=EMBEDDING_DIM, weights=[weights_matrix]))
model.add(Conv1D(FILTERS, KERNEL_SIZE))
model.add(MaxPooling1D(pool_size=POOL_SIZE))
model.add(LSTM(NEURONS, return_sequences=True))
model.add(LSTM(NEURONS))
model.add(Dense(NEURONS, activation='relu'))
model.add(Dense(1, activation='sigmoid'))
model.build(input_shape=(None, 1))
model.summary()


model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

model.fit(X_train, y_train, epochs=10)

In [ ]:
from sklearn.metrics import classification_report


y_pred = model.predict(X_test)
y_pred = (y_pred > 0.5).astype('int')

print(classification_report(y_test, y_pred))